# 00 · Light and light curves

**The big idea:** we can't take a picture of a planet orbiting another star — it's far too small and faint next to its star. But when a planet passes *in front of* its star (a **transit**), it blocks a tiny sliver of light, and the star looks very slightly dimmer for a few hours. If we measure the star's brightness carefully over time, we can *see that dip* even though we can't see the planet.

This notebook builds that idea from scratch using **synthetic data you generate yourself** — no downloads, no astronomy libraries, just `numpy`. Once the shapes are intuitive, the later notebooks repeat the exact same ideas on real telescope data.

**You'll learn:** what "brightness" means to an astronomer · what a light curve is · the geometry of a transit · why *periodicity* is the real signal · how *folding* pulls a signal out of noise.

## 1. How we measure a star's brightness

Two words you'll see everywhere:

- **Flux** — how much light we receive per second. Bigger number = brighter. This is the physical quantity; it's what the code below uses.
- **Magnitude** — an older, logarithmic scale that is *backwards*: **smaller** magnitude = **brighter** star. (A 1st-magnitude star is brighter than a 6th-magnitude one.) You'll meet it in catalogs, but for transit work we mostly stay in flux.

A key move in this whole field is **normalization**: we usually don't care about a star's *absolute* brightness, only how it changes *relative to itself*. So we divide the flux by its typical value, giving a curve that hovers around **1.0**. A 1% transit then shows up as a dip to 0.99.

📖 *Resources:* [Photometry (astronomy)](https://en.wikipedia.org/wiki/Photometry_(astronomy)) · [Apparent magnitude](https://en.wikipedia.org/wiki/Apparent_magnitude)

## 2. A light curve is just brightness vs. time

Point a telescope at a star, measure its (normalized) flux every 30 minutes for months, and plot flux against time. That plot is a **light curve** — the fundamental object of this whole repo.

A perfectly steady star with realistic measurement noise looks like this:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)  # fixed seed so the notebook is reproducible

# 90 days, one measurement every 30 minutes (0.5 hr) -- Kepler's "long cadence"
time = np.arange(0, 90, 0.5/24)          # units: days
noise_level = 0.0005                      # 500 parts-per-million scatter per point
flux = 1.0 + rng.normal(0, noise_level, size=time.size)

plt.figure(figsize=(11, 3))
plt.plot(time, flux, '.', ms=2)
plt.axhline(1.0, color='k', lw=0.8)
plt.xlabel('Time (days)'); plt.ylabel('Normalized flux')
plt.title('A quiet star: flux scattered around 1.0 by measurement noise')
plt.show()

## 3. What a transit looks like

When a planet crosses the star's disk, the dip has a characteristic **trapezoid** shape, described by a few numbers:

- **Depth** — how far the flux drops. This is the headline: geometrically, **depth ≈ (Rₚ / Rₛ)²**, the planet's disk area divided by the star's. A Jupiter-sized planet on a Sun-like star blocks ~1% (depth ≈ 0.01); an Earth blocks ~0.008% (84 ppm) — which is why small planets are *hard*.
- **Duration** — how long the transit lasts (hours), set by the orbital speed and geometry.
- **Ingress / egress** — the sloped sides: the planet takes a little while to fully cross the star's edge, so the dip ramps in and out rather than being a perfect box.

Let's write the model ourselves so there's no magic:

In [ ]:
def transit_model(t, period, t0, duration, depth, ingress_frac=0.1):
    """A simple trapezoidal transit. Returns normalized flux for times t.

    period      : days between transits
    t0          : time of a transit center
    duration    : total transit length (days)
    depth        : fractional dip, ~ (Rp/Rs)**2
    ingress_frac : fraction of the duration spent in ingress/egress ramps
    """
    # fold time so 0 = nearest transit center, range [-period/2, +period/2)
    phase = ((t - t0 + 0.5 * period) % period) - 0.5 * period
    x = np.abs(phase)
    half = duration / 2
    ramp = ingress_frac * duration

    flux = np.ones_like(t)
    flux[x < (half - ramp)] = 1 - depth                       # flat bottom
    edge = (x >= (half - ramp)) & (x < half)                   # sloped sides
    flux[edge] = 1 - depth * (half - x[edge]) / ramp
    return flux

# One clean transit, no noise, so the shape is obvious
t = np.linspace(-0.15, 0.15, 2000)
f = transit_model(t, period=10, t0=0.0, duration=0.12, depth=0.01)

plt.figure(figsize=(7, 4))
plt.plot(t * 24, f)
plt.xlabel('Hours from transit center'); plt.ylabel('Normalized flux')
plt.title('A single transit (depth 1%%  ->  Rp/Rs = sqrt(0.01) = 0.1)')
plt.show()

## 4. Periodicity is the real signal

A single dip is weak evidence — it could be a cosmic ray, a starspot, an instrument glitch, or a passing asteroid. What makes a transit *convincing* is that a planet orbits, so the dip **repeats on a strict period**, forever.

But here's the catch: for a small planet, a single transit can be **shallower than the noise**. If you just plot the full light curve, the dips are invisible to the eye:

In [ ]:
P_true = 3.5           # orbital period (days)
depth_true = 0.0008    # 800 ppm -- only ~1.6x the per-point noise!

signal = transit_model(time, period=P_true, t0=1.3, duration=0.12, depth=depth_true)
flux = signal + rng.normal(0, noise_level, size=time.size)

plt.figure(figsize=(11, 3))
plt.plot(time, flux, '.', ms=2)
plt.xlabel('Time (days)'); plt.ylabel('Normalized flux')
plt.title('The same 90 days, now with a real transiting planet buried in it. See it? Neither can we.')
plt.show()

## 5. Folding: how the signal emerges

Here's the trick that makes exoplanet detection possible. If we know (or guess) the period, we can **fold** the light curve: chop it into period-length chunks and stack them on top of each other. Every transit lands at the same phase and *adds up*, while the random noise partly cancels. Signal-to-noise grows with the number of transits stacked.

Fold on the **true** period and the buried planet snaps into focus:

In [ ]:
def fold(t, period, t0):
    return ((t - t0 + 0.5 * period) % period) - 0.5 * period

phase = fold(time, P_true, 1.3)
order = np.argsort(phase)

plt.figure(figsize=(8, 4))
plt.plot(phase[order] * 24, flux[order], '.', ms=2, alpha=0.4)
# binned average to show the transit clearly
bins = np.linspace(-P_true/2, P_true/2, 120) * 24
idx = np.digitize(phase * 24, bins)
bx = [(bins[i] + bins[i+1]) / 2 for i in range(len(bins)-1)]
by = [np.mean(flux[idx == i+1]) if np.any(idx == i+1) else np.nan for i in range(len(bins)-1)]
plt.plot(bx, by, 'r-', lw=2, label='binned average')
plt.xlim(-4, 4)
plt.xlabel('Hours from transit center'); plt.ylabel('Normalized flux')
plt.title('Folded on the TRUE period -- the transit that was invisible above is now obvious')
plt.legend(); plt.show()

And critically — fold on the **wrong** period and you get mush. This is *why* period-finding works: only the correct period produces a coherent dip. An algorithm can therefore search thousands of trial periods and pick the one that stacks best. That algorithm is **Box Least Squares (BLS)**, which you'll use in notebook 03.

In [ ]:
phase_wrong = fold(time, P_true * 1.13, 1.3)   # 13%% off
order = np.argsort(phase_wrong)

plt.figure(figsize=(8, 4))
plt.plot(phase_wrong[order] * 24, flux[order], '.', ms=2, alpha=0.4)
plt.xlim(-4, 4)
plt.xlabel('Hours from transit center'); plt.ylabel('Normalized flux')
plt.title('Folded on the WRONG period -- the transit smears out into noise')
plt.show()

## Recap

- Astronomers measure a star's **flux** over time; normalized, it hovers at **1.0**.
- A **transit** is a repeating trapezoidal dip; its **depth ≈ (Rₚ/Rₛ)²**.
- A single dip is weak; **periodic repetition** is what makes it a planet.
- **Folding** on the correct period stacks transits above the noise — and *only* the correct period produces a clean dip, which is what lets us *search* for periods automatically.

Everything from here on applies these exact ideas to real Kepler/TESS data.

## Learning resources
- 🌍 [Methods of detecting exoplanets](https://en.wikipedia.org/wiki/Methods_of_detecting_exoplanets) — the transit method in context
- 🌍 [Transit (astronomy)](https://en.wikipedia.org/wiki/Transit_(astronomy))
- 🌍 [Light curve](https://en.wikipedia.org/wiki/Light_curve)
- 🚀 [NASA — Exoplanets: the transit method](https://science.nasa.gov/exoplanets/)
- 📗 [Lightkurve documentation](https://docs.lightkurve.org/) (we start using it next)

**Next:** `01_finding_and_downloading_data.ipynb` — get real light curves from a space telescope.